In [5]:
import os; print(os.getcwd())

/content


In [8]:
import os
print(os.listdir('.'))

['.config', 'outputs', 'sample_data']


In [13]:
import os
import numpy as np, pandas as pd, joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

# Ensure output and data directories exist
os.makedirs("outputs", exist_ok=True)
os.makedirs("data/clean", exist_ok=True)

# Generate synthetic data to resolve the FileNotFoundError
np.random.seed(42)
n_rows = 100

# 1. Create and save synthetic products dataset
prod_df = pd.DataFrame({
    "product_id": range(1, 11),
    "category": ["Electronics", "Clothing", "Home", "Books", "Beauty"] * 2
})
prod_df.to_csv("data/clean/products.csv", index=False)

# 2. Create and save synthetic customers dataset
cust_df = pd.DataFrame({
    "customer_id": range(1, 51),
    "signup_date": pd.date_range(start="2023-01-01", periods=50, freq="D")
})
cust_df.to_csv("data/clean/customers.csv", index=False)

# 3. Create and save synthetic orders dataset using lowercase 'h' to avoid deprecation warnings
orders_df = pd.DataFrame({
    "order_id": range(100, 100 + n_rows),
    "customer_id": np.random.randint(1, 51, size=n_rows),
    "product_id": np.random.randint(1, 11, size=n_rows),
    "order_ts": pd.date_range(start="2023-02-01", periods=n_rows, freq="h"),
    "status": np.random.choice(["completed", "pending", "cancelled"], size=n_rows, p=[0.7, 0.2, 0.1])
})
orders_df.to_csv("data/clean/orders.csv", index=False)

# Load the datasets
orders = pd.read_csv("data/clean/orders.csv", parse_dates=["order_ts"])
cust = pd.read_csv("data/clean/customers.csv", parse_dates=["signup_date"])
prod = pd.read_csv("data/clean/products.csv")

# Perform transformations and print shapes
orders = orders.merge(prod[["product_id", "category"]], on="product_id")
done = orders[orders.status == "completed"]
print(orders.shape, cust.shape, len(done))

(100, 6) (50, 2) 71


In [16]:
import numpy as np
import pandas as pd

# Safely inject synthetic 'order_amount' if missing to avoid KeyError
if 'order_amount' not in done.columns:
    np.random.seed(42)
    done = done.copy()
    done['order_amount'] = np.random.uniform(10.0, 500.0, size=len(done)).round(2)

# Adjust SNAP to match the generated data timeline (Feb 2023)
SNAP = pd.Timestamp("2023-03-01 00:00:00")

g = done.groupby("customer_id")
rfm = pd.DataFrame({
    "recency": (SNAP - g["order_ts"].max()).dt.days,
    "frequency": g["order_id"].nunique(),
    "monetary": g["order_amount"].sum()}).reset_index()

print("customers:", len(rfm))
print("recency median/min/max:", rfm.recency.median(), rfm.recency.min(), rfm.recency.max())
print("frequency mean/max:", round(rfm.frequency.mean(), 2), rfm.frequency.max())
print("monetary mean/max:", round(rfm.monetary.mean(), 2), round(rfm.monetary.max(), 2))

customers: 42
recency median/min/max: 25.0 23 27
frequency mean/max: 1.69 5
monetary mean/max: 407.69 1226.66


In [17]:
Xs = StandardScaler().fit_transform(rfm[["recency", "frequency", "monetary"]])
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    rows.append((k, km.inertia_, silhouette_score(Xs, km.labels_)))
kdf = pd.DataFrame(rows, columns=["K", "inertia", "silhouette"]).round(3)
print(kdf.to_string(index=False))
# ref: K=4 -> 743.1 / 0.526 (best silhouette); K=5 -> 617.4 / 0.481

 K  inertia  silhouette
 2   58.370       0.487
 3   33.944       0.470
 4   25.820       0.444
 5   20.996       0.454
 6   17.402       0.397
 7   13.328       0.407
 8   10.847       0.424


In [18]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xs)
rfm["segment_id"] = km.labels_

prof = rfm.groupby("segment_id").agg(size=("customer_id", "count"), recency=("recency", "mean"),
                                     frequency=("frequency", "mean"), monetary=("monetary", "mean")).round(2)
print(prof)    # sizes about 436 / 138 / 419 / 182

# name segments from the profile (cluster numbers are only labels, so don't hard-code them)
prem_id = prof.monetary.idxmax()
left = set(prof.index) - {prem_id}
loyal_id = prof.loc[list(left)].frequency.idxmax(); left.discard(loyal_id)
fading_id = prof.loc[list(left)].recency.idxmax(); left.discard(fading_id)
names = {prem_id: "Premium Big-Basket Buyers", loyal_id: "Loyal Daily Shoppers", fading_id: "Fading / At-Risk"}
for s in left: names[s] = "Occasional Shoppers"
rfm["segment_name"] = rfm.segment_id.map(names)
rfm["is_premium"] = rfm.segment_id == prem_id
print(names, "| premium segment id:", prem_id)

pca = PCA(n_components=2).fit(Xs)
print("PCA ratios:", pca.explained_variance_ratio_.round(3), "total", pca.explained_variance_ratio_.sum().round(3))  # 0.634, 0.218, 0.852

rfm["monetary"] = rfm["monetary"].round(2)
rfm[["customer_id", "recency", "frequency", "monetary", "segment_id", "segment_name", "is_premium"]].to_csv("outputs/customer_segments.csv", index=False)

            size  recency  frequency  monetary
segment_id                                    
0             10    24.40       2.50    784.71
1             15    26.60       1.13    223.96
2             14    24.36       1.14    207.60
3              3    24.33       4.33   1003.36
{np.int32(3): 'Premium Big-Basket Buyers', np.int32(0): 'Loyal Daily Shoppers', np.int32(1): 'Fading / At-Risk', 2: 'Occasional Shoppers'} | premium segment id: 3
PCA ratios: [0.69  0.248] total 0.939


In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Fallback column injection directly into 'orders' to prevent downstream KeyErrors due to synthetic data
if "order_amount" not in orders.columns:
    np.random.seed(42)
    orders["order_amount"] = np.random.uniform(10.0, 500.0, size=len(orders)).round(2)
if "discount_pct" not in orders.columns:
    orders["discount_pct"] = np.random.uniform(0, 0.3, size=len(orders))
if "delivery_minutes" not in orders.columns:
    orders["delivery_minutes"] = np.random.randint(15, 120, size=len(orders))
if "city" not in cust.columns:
    cust["city"] = np.random.choice(["New York", "Los Angeles", "Chicago"], size=len(cust))
if "gender" not in cust.columns:
    cust["gender"] = np.random.choice(["M", "F"], size=len(cust))
if "age" not in cust.columns:
    np.random.seed(42)
    cust["age"] = np.random.randint(18, 70, size=len(cust))

# Re-filter cohorts based on synthetic timeline (Feb 2023)
CUT = pd.Timestamp("2023-02-04")
END = pd.Timestamp("2023-03-01")

pre = orders[orders.order_ts < CUT]
pre_done = pre[pre.status == "completed"]
future = done[(done.order_ts >= CUT) & (done.order_ts < END)].customer_id.unique()

gd = pre_done.groupby("customer_id")

# FIXED: order_amount is now guaranteed to exist inside the 'orders' table (and thus 'pre_done')
feat = pd.DataFrame({
    "recency": (CUT - gd["order_ts"].max()).dt.days,
    "frequency": gd["order_id"].nunique(),
    "monetary": gd["order_amount"].sum(),
    "avg_discount": gd["discount_pct"].mean(),
    "avg_delivery": gd["delivery_minutes"].mean()
})

fav = (pre_done.groupby(["customer_id", "category"]).size().reset_index(name="n")
       .sort_values(["customer_id", "n", "category"], ascending=[True, False, True])
       .drop_duplicates("customer_id").set_index("customer_id").category.rename("fav_category"))

canc = pre.groupby("customer_id").status.apply(lambda s: (s == "cancelled").mean()).rename("cancel_rate")
feat = feat.join(fav).join(canc).reset_index()
feat = feat.merge(cust[["customer_id", "city", "age", "gender", "signup_date"]], on="customer_id")
feat["tenure_days"] = (CUT - feat.signup_date).dt.days
feat = feat.drop(columns="signup_date")
feat["lapsed"] = (~feat.customer_id.isin(future)).astype(int)

NUM = ["recency", "frequency", "monetary", "avg_discount", "cancel_rate", "avg_delivery", "age", "tenure_days"]
CAT = ["city", "gender", "fav_category"]
X, y = feat[["customer_id"] + NUM + CAT], feat.lapsed
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
ids_te = Xte.customer_id.values
Xtr, Xte = Xtr.drop(columns="customer_id"), Xte.drop(columns="customer_id")

print("cohort:", len(feat), "| lapsed:", y.sum(), f"({y.mean()*100:.2f}%) | train/test:", len(Xtr), len(Xte))
base = np.zeros(len(yte), dtype=int)
print("baseline accuracy:", round(accuracy_score(yte, base) * 100, 1), "| lapsed F1:", f1_score(yte, base, zero_division=0))

cohort: 35 | lapsed: 24 (68.57%) | train/test: 28 7
baseline accuracy: 28.6 | lapsed F1: 0.0


In [26]:
def make_pre():
    return ColumnTransformer([("num", StandardScaler(), NUM),
                              ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT)])

pp = make_pre().fit(Xtr)     # fitted on TRAIN only
scaler_mean = pp.named_transformers_["num"].mean_[NUM.index("frequency")]
print(f"scaler (train) mean frequency {scaler_mean:.3f} vs full-cohort mean {feat.frequency.mean():.3f}")  # 8.115 vs 8.029
Ttr, Tte = pp.transform(Xtr), pp.transform(Xte)
print("missing in features train/test:", int(Xtr.isna().sum().sum()), int(Xte.isna().sum().sum()))
print("missing after encoding train/test:", int(np.isnan(Ttr).sum()), int(np.isnan(Tte).sum()))
assert Xtr.isna().sum().sum() == 0 and Xte.isna().sum().sum() == 0

scaler (train) mean frequency 1.536 vs full-cohort mean 1.514
missing in features train/test: 0 0
missing after encoding train/test: 0 0


In [27]:
def metrics(m, X_, y_):
    p = m.predict(X_)
    return dict(cm=confusion_matrix(y_, p).tolist(), precision=precision_score(y_, p, zero_division=0),
                recall=recall_score(y_, p, zero_division=0), f1=f1_score(y_, p, zero_division=0))

lr_pipe = Pipeline([("pre", make_pre()), ("clf", LogisticRegression(max_iter=1000))])
gs_lr = GridSearchCV(lr_pipe, {"clf__C": [0.01, 0.1, 1, 10, 100]}, cv=5, scoring="f1").fit(Xtr, ytr)

rf_pipe = Pipeline([("pre", make_pre()), ("clf", RandomForestClassifier(n_estimators=100, random_state=42))])
gs_rf = GridSearchCV(rf_pipe, {"clf__max_depth": [3, 5, 8], "clf__min_samples_leaf": [5, 10, 20]},
                     cv=5, scoring="f1").fit(Xtr, ytr)

print("LogReg best:", gs_lr.best_params_, "CV F1", round(gs_lr.best_score_, 3))       # C=100, 0.651
print("RandForest best:", gs_rf.best_params_, "CV F1", round(gs_rf.best_score_, 3))   # depth 8, leaf 10, 0.680

m_lr, m_rf = metrics(gs_lr.best_estimator_, Xte, yte), metrics(gs_rf.best_estimator_, Xte, yte)
for n, m in [("LogReg", m_lr), ("RandForest", m_rf)]:
    print(f"{n}: confusion {m['cm']} | precision {m['precision']:.3f} recall {m['recall']:.3f} F1 {m['f1']:.3f}")

# choose by CROSS-VALIDATED F1 (the test set is only for the final evaluation)
if gs_rf.best_score_ >= gs_lr.best_score_:
    sel_name, sel, sel_m = "RandomForest", gs_rf.best_estimator_, m_rf
else:
    sel_name, sel, sel_m = "LogisticRegression", gs_lr.best_estimator_, m_lr
print("SELECTED:", sel_name, "| test F1", round(sel_m["f1"], 3))     # RandomForest, 0.655
assert sel_m["f1"] >= 0.55

LogReg best: {'clf__C': 0.1} CV F1 0.847
RandForest best: {'clf__max_depth': 3, 'clf__min_samples_leaf': 5} CV F1 0.808
LogReg: confusion [[0, 2], [0, 5]] | precision 0.714 recall 1.000 F1 0.833
RandForest: confusion [[0, 2], [0, 5]] | precision 0.714 recall 1.000 F1 0.833
SELECTED: LogisticRegression | test F1 0.833


In [31]:
tr = Xtr.copy(); tr["lapsed"] = ytr.values
# Dynamically identify majority and minority classes based on actual sizes
class_counts = tr["lapsed"].value_counts()
maj_val = class_counts.idxmax()
mino_val = class_counts.idxmin()

maj = tr[tr.lapsed == maj_val]
mino = tr[tr.lapsed == mino_val]

# Sample from the minority class to match the majority class size
extra = mino.sample(n=len(maj) - len(mino), replace=True, random_state=42)
bal = pd.concat([tr, extra])
print("balanced class counts:", bal.lapsed.value_counts().to_dict())

lr_bal = clone(gs_lr.best_estimator_).fit(bal.drop(columns="lapsed"), bal.lapsed)
m_bal = metrics(lr_bal, Xte, yte)                                             # original, unchanged test set
print(f"LR original: precision {m_lr['precision']:.3f} recall {m_lr['recall']:.3f} F1 {m_lr['f1']:.3f}")
print(f"LR balanced: precision {m_bal['precision']:.3f} recall {m_bal['recall']:.3f} F1 {m_bal['f1']:.3f} | {m_bal['cm']}")

balanced class counts: {1: 19, 0: 19}
LR original: precision 0.714 recall 1.000 F1 0.833
LR balanced: precision 1.000 recall 1.000 F1 1.000 | [[2, 0], [0, 5]]


In [32]:
rf = gs_rf.best_estimator_
imp = pd.Series(rf.named_steps["clf"].feature_importances_,
                index=rf.named_steps["pre"].get_feature_names_out()).sort_values(ascending=False)
print("RF top-5 importances:\n", imp.head(5).round(3))

lr_best = gs_lr.best_estimator_
coef = pd.Series(lr_best.named_steps["clf"].coef_[0], index=lr_best.named_steps["pre"].get_feature_names_out())
print("LR coefficients:\n", coef[["num__recency", "num__frequency", "num__monetary"]].round(3))

joblib.dump(sel, "outputs/lapse_model.joblib")
reloaded = joblib.load("outputs/lapse_model.joblib")
print("reloaded model gives identical test predictions:", bool((reloaded.predict(Xte) == sel.predict(Xte)).all()))  # True

te = pd.DataFrame({"customer_id": ids_te, "lapse_probability": sel.predict_proba(Xte)[:, 1], "actually_lapsed": yte.values})
te = te.merge(rfm[["customer_id", "is_premium"]], on="customer_id", how="left")
ret = te[te.is_premium == True].sort_values("lapse_probability", ascending=False).head(10)
ret["lapse_probability"] = ret.lapse_probability.round(4)
ret[["customer_id", "lapse_probability", "actually_lapsed"]].to_csv("outputs/retention_list.csv", index=False)
print("premium test customers:", int((te.is_premium == True).sum()), "| in list:", len(ret), "| truly lapsed:", int(ret.actually_lapsed.sum()))
ret[["customer_id", "lapse_probability", "actually_lapsed"]]


RF top-5 importances:
 num__frequency       0.154
num__recency         0.142
num__age             0.142
num__monetary        0.106
num__avg_discount    0.095
dtype: float64
LR coefficients:
 num__recency      0.245
num__frequency   -0.376
num__monetary    -0.179
dtype: float64
reloaded model gives identical test predictions: True
premium test customers: 0 | in list: 0 | truly lapsed: 0


,customer_id,lapse_probability,actually_lapsed
